# 豆瓣 Top250 电影数据采集

这个 Notebook 使用 `requests + BeautifulSoup` 抓取 [豆瓣电影 Top250](https://movie.douban.com/top250) 的列表页与详情页数据。

目标字段：

- `rank`
- `title_cn`
- `title_other`
- `year`
- `rating`
- `rating_count`
- `quote`
- `director`
- `screenwriter`
- `actors`
- `genres`
- `country_region`
- `language`
- `release_dates`
- `runtime_minutes`
- `imdb`
- `detail_url`
- `poster_url`
- `crawl_time`

说明：

- 为了减少请求压力，代码里加入了随机等待。
- 如果网络波动或页面结构变化，个别字段可能为空，需要按实际页面微调选择器。

In [12]:
# %pip install requests beautifulsoup4 pandas openpyxl lxml

In [13]:
import random
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib.parse import urljoin
from urllib3.util.retry import Retry

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

BASE_URL = 'https://movie.douban.com/top250'
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Referer': 'https://movie.douban.com/',
    'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8'
}
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

In [14]:
def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=2,
        backoff_factor=1,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=['GET']
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    session.headers.update(HEADERS)
    return session


def fetch_html(
    session: requests.Session,
    url: str,
    params: dict | None = None,
    max_retries: int = 4,
    base_sleep: float = 8
) -> str:
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(url, params=params, timeout=20, allow_redirects=False)

            if response.status_code in {301, 302, 303, 307, 308}:
                location = response.headers.get('Location', '')
                redirect_url = urljoin(url, location)

                if 'sec.douban.com' in redirect_url:
                    wait_seconds = base_sleep * attempt + random.uniform(2, 5)
                    print(f'     命中风控，等待 {wait_seconds:.1f}s 后重试: {url}')
                    time.sleep(wait_seconds)
                    continue

                response = session.get(redirect_url, timeout=20)

            if response.status_code == 429:
                wait_seconds = base_sleep * attempt + random.uniform(2, 5)
                print(f'     遇到 429，等待 {wait_seconds:.1f}s 后重试: {url}')
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            return response.text

        except requests.RequestException as exc:
            last_error = exc
            wait_seconds = min(base_sleep * attempt, 30) + random.uniform(1, 3)
            print(f'     请求异常，第 {attempt} 次重试前等待 {wait_seconds:.1f}s: {exc}')
            time.sleep(wait_seconds)

    raise RuntimeError(f'抓取失败，已重试 {max_retries} 次: {url}') from last_error


def clean_text(text: str | None) -> str:
    if not text:
        return ''
    return re.sub(r'\s+', ' ', text).strip()


def normalize_label(text: str) -> str:
    return re.sub(r'[:：\s]+', '', text or '')


def extract_labeled_text(info_block, label: str) -> str:
    target = normalize_label(label)
    label_node = info_block.find(
        'span',
        class_='pl',
        string=lambda s: s and normalize_label(s) == target
    )
    if not label_node:
        return ''

    parts = []
    for sibling in label_node.next_siblings:
        name = getattr(sibling, 'name', None)
        if name == 'br' or (name == 'span' and 'pl' in (sibling.get('class') or [])):
            break
        text = sibling.get_text(' ', strip=True) if hasattr(sibling, 'get_text') else str(sibling).strip()
        if text:
            parts.append(text)

    return clean_text(' '.join(parts).replace('/', ' / '))


def parse_runtime_minutes(runtime_text: str) -> int | None:
    match = re.search(r'(\d+)', runtime_text or '')
    return int(match.group(1)) if match else None


def safe_float(text: str) -> float | None:
    try:
        return float(text)
    except (TypeError, ValueError):
        return None


def safe_int(text: str) -> int | None:
    digits = re.sub(r'[^\d]', '', text or '')
    return int(digits) if digits else None

In [15]:
def parse_top250_page(html: str) -> list[dict]:
    soup = BeautifulSoup(html, 'html.parser')
    movies = []

    for item in soup.select('ol.grid_view > li'):
        rank = clean_text(item.select_one('em').get_text())
        title_nodes = item.select('span.title')
        other_nodes = item.select('span.other')
        title_cn = clean_text(title_nodes[0].get_text()) if title_nodes else ''
        title_other_parts = [
            clean_text(node.get_text()).lstrip('/').strip()
            for node in title_nodes[1:] + other_nodes
            if clean_text(node.get_text())
        ]
        title_other = ' / '.join(
            part for part in title_other_parts if part
        )
        rating = clean_text(item.select_one('span.rating_num').get_text()) if item.select_one('span.rating_num') else ''
        rating_count = clean_text(item.select('div.star span')[-1].get_text()) if item.select('div.star span') else ''
        quote_node = item.select_one('span.inq')
        quote = clean_text(quote_node.get_text()) if quote_node else ''
        link_node = item.select_one('div.hd a')
        poster_node = item.select_one('div.pic img')

        movies.append({
            'rank': safe_int(rank),
            'title_cn': title_cn,
            'title_other': title_other,
            'rating': safe_float(rating),
            'rating_count': safe_int(rating_count),
            'quote': quote,
            'detail_url': link_node['href'] if link_node else '',
            'poster_url': poster_node['src'] if poster_node else ''
        })

    return movies

In [16]:
def parse_movie_detail(html: str) -> dict:
    soup = BeautifulSoup(html, 'html.parser')
    info = soup.select_one('#info')
    if not info:
        return {}

    title_text = clean_text(soup.select_one('span[property="v:itemreviewed"]').get_text()) if soup.select_one('span[property="v:itemreviewed"]') else ''
    year_text = clean_text(soup.select_one('#content span.year').get_text()) if soup.select_one('#content span.year') else ''
    year_match = re.search(r'(\d{4})', year_text)

    director = ' / '.join(clean_text(x.get_text()) for x in info.select('a[rel="v:directedBy"]'))
    actors = ' / '.join(clean_text(x.get_text()) for x in info.select('a[rel="v:starring"]'))
    genres = ' / '.join(clean_text(x.get_text()) for x in info.select('span[property="v:genre"]'))
    release_dates = ' / '.join(clean_text(x.get_text()) for x in info.select('span[property="v:initialReleaseDate"]'))
    runtime_text = clean_text(info.select_one('span[property="v:runtime"]').get_text()) if info.select_one('span[property="v:runtime"]') else ''
    rating = clean_text(soup.select_one('strong[property="v:average"]').get_text()) if soup.select_one('strong[property="v:average"]') else ''
    rating_count = clean_text(soup.select_one('span[property="v:votes"]').get_text()) if soup.select_one('span[property="v:votes"]') else ''

    return {
        'detail_title': title_text,
        'year': int(year_match.group(1)) if year_match else None,
        'director': director,
        'screenwriter': extract_labeled_text(info, '编剧'),
        'actors': actors,
        'genres': genres,
        'country_region': extract_labeled_text(info, '制片国家/地区'),
        'language': extract_labeled_text(info, '语言'),
        'release_dates': release_dates,
        'runtime': runtime_text,
        'runtime_minutes': parse_runtime_minutes(runtime_text),
        'aka': extract_labeled_text(info, '又名'),
        'imdb': extract_labeled_text(info, 'IMDb'),
        'rating': safe_float(rating),
        'rating_count': safe_int(rating_count)
    }

In [17]:
def crawl_douban_top250_list_only(
    list_delay_range: tuple[float, float] = (2.0, 4.0)
) -> pd.DataFrame:
    session = build_session()
    records = []

    for start in range(0, 250, 25):
        print(f'正在抓取列表页: start={start}')
        html = fetch_html(session, BASE_URL, params={'start': start}, max_retries=3, base_sleep=10)
        page_movies = parse_top250_page(html)

        for movie in page_movies:
            movie['crawl_time'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            records.append(movie)

        time.sleep(random.uniform(*list_delay_range))

    df = pd.DataFrame(records)
    column_order = [
        'rank', 'title_cn', 'title_other', 'year', 'rating', 'rating_count',
        'quote', 'detail_url', 'poster_url', 'crawl_time'
    ]
    existing_columns = [col for col in column_order if col in df.columns]
    return df[existing_columns]


def enrich_movie_details(
    df_movies: pd.DataFrame,
    max_movies: int | None = 10,
    detail_delay_range: tuple[float, float] = (8.0, 15.0),
    checkpoint_every: int = 5
) -> pd.DataFrame:
    session = build_session()
    records = []

    if max_movies is not None:
        source_df = df_movies.head(max_movies).copy()
    else:
        source_df = df_movies.copy()

    for idx, row in source_df.iterrows():
        movie = row.to_dict()
        detail_url = movie.get('detail_url', '')
        print(f"  -> 详情页: {movie.get('rank')} {movie.get('title_cn')}")

        try:
            detail_html = fetch_html(session, detail_url, max_retries=4, base_sleep=12)
            detail_data = parse_movie_detail(detail_html)
        except Exception as exc:
            print(f'     详情页抓取失败: {detail_url} | {exc}')
            detail_data = {}

        merged = {**movie, **detail_data}
        records.append(merged)

        if checkpoint_every and len(records) % checkpoint_every == 0:
            checkpoint_df = pd.DataFrame(records)
            checkpoint_path = OUTPUT_DIR / 'douban_top250_detail_checkpoint.csv'
            checkpoint_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
            print(f'     已保存详情检查点: {checkpoint_path.resolve()}')

        time.sleep(random.uniform(*detail_delay_range))

    return pd.DataFrame(records)


def crawl_douban_top250(
    max_movies: int | None = 50,
    list_delay_range: tuple[float, float] = (2.0, 4.0),
    detail_delay_range: tuple[float, float] = (5.0, 9.0),
    checkpoint_every: int = 20
) -> pd.DataFrame:
    session = build_session()
    records = []
    collected = 0

    for start in range(0, 250, 25):
        print(f'正在抓取列表页: start={start}')
        html = fetch_html(session, BASE_URL, params={'start': start}, max_retries=3, base_sleep=10)
        page_movies = parse_top250_page(html)

        for movie in page_movies:
            if max_movies is not None and collected >= max_movies:
                break

            detail_url = movie.get('detail_url', '')
            print(f"  -> 详情页: {movie.get('rank')} {movie.get('title_cn')}")

            try:
                detail_html = fetch_html(session, detail_url, max_retries=4, base_sleep=12)
                detail_data = parse_movie_detail(detail_html)
            except Exception as exc:
                print(f'     详情页抓取失败: {detail_url} | {exc}')
                detail_data = {}

            merged = {**movie, **detail_data}
            merged['crawl_time'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            if not merged.get('title_cn') and merged.get('detail_title'):
                merged['title_cn'] = merged['detail_title']

            records.append(merged)
            collected += 1

            if checkpoint_every and collected % checkpoint_every == 0:
                checkpoint_df = pd.DataFrame(records)
                checkpoint_path = OUTPUT_DIR / 'douban_top250_checkpoint.csv'
                checkpoint_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
                print(f'     已保存检查点: {checkpoint_path.resolve()}')

            time.sleep(random.uniform(*detail_delay_range))

        if max_movies is not None and collected >= max_movies:
            break

        time.sleep(random.uniform(*list_delay_range))

    df = pd.DataFrame(records)

    column_order = [
        'rank', 'title_cn', 'title_other', 'detail_title', 'aka', 'year', 'rating', 'rating_count',
        'quote', 'director', 'screenwriter', 'actors', 'genres', 'country_region', 'language',
        'release_dates', 'runtime', 'runtime_minutes', 'imdb', 'detail_url', 'poster_url', 'crawl_time'
    ]
    existing_columns = [col for col in column_order if col in df.columns]
    return df[existing_columns]

In [18]:
df_movies = crawl_douban_top250_list_only()
print(f'共抓取 {len(df_movies)} 条记录')
df_movies.head()

正在抓取列表页: start=0
正在抓取列表页: start=25
正在抓取列表页: start=50
正在抓取列表页: start=75
正在抓取列表页: start=100
正在抓取列表页: start=125
正在抓取列表页: start=150
正在抓取列表页: start=175
正在抓取列表页: start=200
正在抓取列表页: start=225
共抓取 250 条记录


,rank,title_cn,title_other,rating,rating_count,quote,detail_url,poster_url,crawl_time
0,1,肖申克的救赎,The Shawshank Redemption / 月黑高飞(港) / 刺激1995(台),9.7,None,,https://movie.douban.com/subject/1292052/,https://img3.doubanio.com/view/photo/s_ratio_p...,2026-05-19 22:52:02
1,2,霸王别姬,再见，我的妾 / Farewell My Concubine,9.6,None,,https://movie.douban.com/subject/1291546/,https://img1.doubanio.com/view/photo/s_ratio_p...,2026-05-19 22:52:02
2,3,泰坦尼克号,Titanic / 铁达尼号(港 / 台),9.5,None,,https://movie.douban.com/subject/1292722/,https://img9.doubanio.com/view/photo/s_ratio_p...,2026-05-19 22:52:02
3,4,阿甘正传,Forrest Gump / 福雷斯特·冈普,9.5,None,,https://movie.douban.com/subject/1292720/,https://img3.doubanio.com/view/photo/s_ratio_p...,2026-05-19 22:52:02
4,5,千与千寻,千と千尋の神隠し / 神隐少女(台) / 千与千寻的神隐,9.4,None,,https://movie.douban.com/subject/1291561/,https://img1.doubanio.com/view/photo/s_ratio_p...,2026-05-19 22:52:02


In [19]:
csv_path = OUTPUT_DIR / 'douban_top250_movies_list.csv'
excel_path = OUTPUT_DIR / 'douban_top250_movies_list.xlsx'

df_movies.to_csv(csv_path, index=False, encoding='utf-8-sig')
df_movies.to_excel(excel_path, index=False)

print(f'CSV 已保存: {csv_path.resolve()}')
print(f'Excel 已保存: {excel_path.resolve()}')

CSV 已保存: D:\AI\App01\output\douban_top250_movies_list.csv
Excel 已保存: D:\AI\App01\output\douban_top250_movies_list.xlsx


In [20]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rank          250 non-null    int64  
 1   title_cn      250 non-null    object 
 2   title_other   250 non-null    object 
 3   rating        250 non-null    float64
 4   rating_count  0 non-null      object 
 5   quote         250 non-null    object 
 6   detail_url    250 non-null    object 
 7   poster_url    250 non-null    object 
 8   crawl_time    250 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 17.7+ KB
